# Lab 2B — Swap the Retriever, Change Nothing Else

[Lab 2A](#) claimed `core.py` was portable and proved it by grep. This lab proves
it the expensive way: by **replacing an entire adapter** and seeing whether the
core notices.

Then it asks a harder question — *did the swap actually make anything better?*
The answer is not what most people expect, and it is the reason
[Session 6](#) exists.

| Cell group | What you do |
|---|---|
| 0–2 | Rebuild the Lab 2A setup: docs, core, chat, keyword retriever. |
| 3 | Write a **second** retriever using embeddings. |
| 4 | Run the same question through both. |
| 5 | Compare the **answers**. |
| 6 | Compare what they **retrieved**. |
| 7 | The uncomfortable finding. |

**Compute:** Serverless. **Prerequisite:** run [Lab 2A](#) first.

In [0]:
%pip install -q openai
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


## 1. Rebuild the Lab 2A setup

Same documents, same `core.py`, same chat adapter, same keyword retriever. Nothing
here is new — it is cell 1 through 5 of Lab 2A, condensed.

In [0]:
import math, re, json, warnings
from collections import Counter
from databricks.sdk import WorkspaceClient
from openai import OpenAI

warnings.filterwarnings("ignore", message=".*serializer warnings.*")

docs = [r.asDict() for r in spark.sql("""
    SELECT doc_id, title, category, audience, body
    FROM agents_labs.retail.support_docs ORDER BY doc_id
""").collect()]

w = WorkspaceClient()
_token = w.config.authenticate()["Authorization"].removeprefix("Bearer ")
_client = OpenAI(api_key=_token, base_url=f"{w.config.host}/serving-endpoints")


class DatabricksChat:
    def __init__(self, model="databricks-claude-sonnet-5"):
        self._model = model
    @property
    def model(self):
        return self._model
    def complete(self, messages, tools):
        return _client.chat.completions.create(
            model=self._model, messages=messages, tools=tools, max_tokens=900
        ).choices[0].message


_WORD = re.compile(r"[a-z0-9]+")
def _tok(s): return _WORD.findall(s.lower())


class KeywordRetriever:
    name = "keyword-tfidf"
    def __init__(self, docs):
        self.docs = docs
        self._tf = [Counter(_tok(f"{d['title']} {d['body']}")) for d in docs]
        df = Counter()
        for t in self._tf: df.update(set(t))
        n = len(docs)
        self._idf = {w: math.log((n + 1) / (c + 0.5)) for w, c in df.items()}
    def _score(self, q, tf):
        if not tf: return 0.0
        num = sum(tf[w] * self._idf.get(w, 0.0) for w in q)
        return num / (math.sqrt(sum(v*v for v in tf.values())) or 1.0)
    def search(self, query, k=3, **filters):
        aud = filters.get("audience")
        q = _tok(query)
        out = [{"doc_id": d["doc_id"], "title": d["title"], "text": d["body"],
                "score": self._score(q, tf)}
               for d, tf in zip(self.docs, self._tf) if not aud or d["audience"] == aud]
        out.sort(key=lambda r: r["score"], reverse=True)
        return out[:k]


class SparkOrderTools:
    def schemas(self):
        return [{"type": "function", "function": {
            "name": "get_order",
            "description": "Look up one order and the customer who placed it: status, "
                           "item, revenue, the customer's region and loyalty tier.",
            "parameters": {"type": "object",
                           "properties": {"order_id": {"type": "string"}},
                           "required": ["order_id"]}}}]
    def call(self, name, args):
        oid = str(args.get("order_id", "")).replace("'", "")
        rows = spark.sql(f"""
            SELECT o.order_id, o.status, o.item, o.units, o.revenue, o.order_date,
                   c.name AS customer, c.region, c.tier
            FROM agents_labs.retail.orders o
            JOIN agents_labs.retail.customers c USING (customer_id)
            WHERE o.order_id = '{oid}'""").collect()
        return rows[0].asDict() if rows else {"error": f"no such order: {oid}"}


print(f"  {len(docs)} documents, chat + keyword retriever + order tools ready")

  7 documents, chat + keyword retriever + order tools ready


## 2. Bring back `core.py` — byte for byte

This notebook stands alone, so it recreates `core.py` rather than depending on
Lab 2A's session. **It must be the identical file** — otherwise "we changed
nothing" is not a claim you can make.

The cell prints a hash. Compare it with the one Lab 2A printed.

In [0]:
import os, textwrap, hashlib

CORE_SRC = r'''"""The portable core. Knows nothing about Databricks, or any model."""
from __future__ import annotations
import json
from typing import Protocol, Any


class Retriever(Protocol):
    def search(self, query: str, k: int = 3, **filters: Any) -> list[dict]: ...
    @property
    def name(self) -> str: ...


class ToolBox(Protocol):
    def schemas(self) -> list[dict]: ...
    def call(self, name: str, args: dict) -> dict: ...


class Chat(Protocol):
    def complete(self, messages: list[dict], tools: list[dict] | None) -> Any: ...
    @property
    def model(self) -> str: ...


SYSTEM = """You are a customer support agent for an office furniture retailer.

You have two sources and you usually need both:
  * search_policy  - the company's written policies (delivery, returns, warranty)
  * the order tools - live facts about a specific order and its customer

Never state a policy you have not retrieved, and never state an order fact you
have not looked up. If answering needs both, use both before you answer.
Cite the policy document id you relied on, like [DOC-003]."""

MAX_STEPS = 6


def text_of(message) -> str:
    """Claude endpoints return content as a list of typed blocks, not a string."""
    c = getattr(message, "content", None)
    if c is None:
        return ""
    if isinstance(c, str):
        return c
    out = []
    for b in c:
        b = b if isinstance(b, dict) else getattr(b, "__dict__", {})
        if b.get("type") == "text" and b.get("text"):
            out.append(b["text"])
    return "\n".join(out).strip()


def run(question: str, *, chat: Chat, retriever: Retriever, tools: ToolBox,
        verbose: bool = True) -> dict:
    """One turn of the agent. Returns the answer plus which sources it touched."""
    schemas = [{
        "type": "function",
        "function": {
            "name": "search_policy",
            "description": "Search the company policy documents. Use for anything "
                           "about delivery times, returns, refunds, warranty or billing.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string"},
                    "audience": {"type": "string",
                                 "description": "customer or agent_only. Omit for customer."},
                },
                "required": ["query"],
            },
        },
    }] + tools.schemas()

    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": question}]
    used = {"retrieval": [], "tools": []}

    for step in range(1, MAX_STEPS + 1):
        msg = chat.complete(messages, schemas)
        messages.append(msg.model_dump(exclude_none=True))

        if not msg.tool_calls:
            if verbose:
                print(f"  step {step}: answer")
            return {"answer": text_of(msg), "used": used,
                    "retriever": retriever.name, "model": chat.model}

        for tc in msg.tool_calls:
            name, args = tc.function.name, json.loads(tc.function.arguments or "{}")
            if name == "search_policy":
                hits = retriever.search(args["query"], k=3,
                                        audience=args.get("audience", "customer"))
                used["retrieval"] += [h["doc_id"] for h in hits]
                result = {"results": [{"doc_id": h["doc_id"], "title": h["title"],
                                       "text": h["text"], "score": round(h["score"], 4)}
                                      for h in hits]}
                if verbose:
                    print(f"  step {step}: search_policy({args['query']!r}) "
                          f"-> {[h['doc_id'] for h in hits]}")
            else:
                result = tools.call(name, args)
                used["tools"].append(name)
                if verbose:
                    print(f"  step {step}: {name}({json.dumps(args)}) -> "
                          f"{json.dumps(result, default=str)[:110]}")
            messages.append({"role": "tool", "tool_call_id": tc.id,
                             "content": json.dumps(result, default=str)})

    return {"answer": "(no answer within MAX_STEPS)", "used": used,
            "retriever": retriever.name, "model": chat.model}
'''

with open("core.py", "w") as f:
    f.write(CORE_SRC)

src = open("core.py").read()
print(f"  core.py    : {len(src.splitlines())} lines")
print(f"  imports    : {re.findall(r'^(?:from|import) +([a-z_.]+)', src, re.M)}")
print(f"  sha256[:16]: {hashlib.sha256(src.encode()).hexdigest()[:16]}")
print("\n  Compare that hash with Lab 2A cell 3. Same file, about to be driven")
print("  by a completely different retriever.")

  core.py    : 109 lines
  imports    : ['__future__', 'json', 'typing']
  sha256[:16]: dd0339d663883037

  Compare that hash with Lab 2A cell 3. Same file, about to be driven
  by a completely different retriever.


## 3. A completely different retriever

The keyword retriever matches **words**. This one matches **meaning** — it embeds
every document once, embeds the query, and ranks by cosine similarity.

Shared with `KeywordRetriever`: **nothing** except the two methods `core.py` calls.

```python
    search(query, k, **filters) -> list[dict]
    name -> str
```

In [0]:
def _cos(a, b):
    dot = sum(x*y for x, y in zip(a, b))
    na = math.sqrt(sum(x*x for x in a)) or 1.0
    nb = math.sqrt(sum(y*y for y in b)) or 1.0
    return dot / (na * nb)


class EmbeddingRetriever:
    name = "databricks-embeddings"

    def __init__(self, docs, model="databricks-gte-large-en"):
        self.docs = docs
        self.model = model
        self._vecs = self._embed([f"{d['title']}. {d['body']}" for d in docs])

    def _embed(self, texts):
        r = _client.embeddings.create(model=self.model, input=texts)
        return [d.embedding for d in r.data]

    def search(self, query, k=3, **filters):
        aud = filters.get("audience")
        qv = self._embed([query])[0]
        out = [{"doc_id": d["doc_id"], "title": d["title"], "text": d["body"],
                "score": _cos(qv, v)}
               for d, v in zip(self.docs, self._vecs) if not aud or d["audience"] == aud]
        out.sort(key=lambda r: r["score"], reverse=True)
        return out[:k]


kw = KeywordRetriever(docs)
emb = EmbeddingRetriever(docs)
tools = SparkOrderTools()
chat = DatabricksChat()

print(f"  {kw.name:26} ready")
print(f"  {emb.name:26} ready  ({len(emb._vecs)} vectors, dim {len(emb._vecs[0])})")

  keyword-tfidf              ready
  databricks-embeddings      ready  (7 vectors, dim 1024)


## 4. The swap

The same question, the same `core.run`, the same chat adapter, the same tools.
**One argument changes.**

```python
    core.run(q, chat=chat, retriever=kw,  tools=tools)   # before
    core.run(q, chat=chat, retriever=emb, tools=tools)   # after
```

In [0]:
import core

Q = "My order ORD-1044 hasn't arrived. How long should it take, and what are my options?"

print("=" * 74, "\n  KEYWORD\n", "=" * 74)
a = core.run(Q, chat=chat, retriever=kw, tools=tools)

print("\n" + "=" * 74, "\n  EMBEDDINGS\n", "=" * 74)
b = core.run(Q, chat=chat, retriever=emb, tools=tools)

print("\n  core.py required no changes.")

  KEYWORD
  step 1: get_order({"order_id": "ORD-1044"}) -> {"order_id": "ORD-1044", "status": "delivered", "item": "standing desk", "units": 5, "revenue": "2700.00", "or
  step 1: search_policy('delivery time estimate late delivery options') -> ['DOC-003', 'DOC-001', 'DOC-004']
  step 2: answer

  EMBEDDINGS
  step 1: get_order({"order_id": "ORD-1044"}) -> {"order_id": "ORD-1044", "status": "delivered", "item": "standing desk", "units": 5, "revenue": "2700.00", "or
  step 1: search_policy('delivery time estimate and options for delayed orders') -> ['DOC-003', 'DOC-007', 'DOC-005']
  step 2: answer

  core.py required no changes.


## 5. Compare the answers

This is the part worth slowing down for.

In [0]:
for label, r in (("KEYWORD", a), ("EMBEDDINGS", b)):
    print("=" * 74)
    print(f"  {label}   retriever={r['retriever']}")
    print("=" * 74)
    print(textwrap.fill(r["answer"], 72, initial_indent="  ",
                        subsequent_indent="  "))
    print()

  KEYWORD   retriever=keyword-tfidf
  Good news — I checked your order, and ORD-1044 (5 standing desks for
  Helsinki Works) is actually already marked as **delivered** in our
  system, not in transit.  For reference on timing: deliveries to the
  Nordics normally take **5–7 working days**, since shipments are
  consolidated at our Hamburg hub before final transport [DOC-003].
  Since our records show it as delivered, here's what I'd suggest:  1.
  **Double-check with your team/reception** — sometimes deliveries are
  received by someone else on-site or left in a mailroom/loading dock.
  2. **Check your delivery confirmation email** for the exact delivery
  date and any signature/photo proof. 3. **If it genuinely hasn't
  arrived**, please let me know and I can escalate this as a lost
  shipment investigation — this is different from a standard return.
  Also worth knowing for future orders: if any item ever arrives
  **damaged**, our standard 30-day return window doesn't apply — you'r

## 6. Now compare what they *retrieved*

The answers above probably looked equally good. The retrieval did not.

In [0]:
print(f"  keyword    documents: {a['used']['retrieval']}")
print(f"  embeddings documents: {b['used']['retrieval']}")
print(f"  same set? {set(a['used']['retrieval']) == set(b['used']['retrieval'])}\n")

probe = "how long does delivery to the nordics take"
print(f"  probe: {probe!r}\n")
for r in (kw, emb):
    print(f"  {r.name}")
    for h in r.search(probe, k=3, audience="customer"):
        print(f"      {h['doc_id']}  score={h['score']:.4f}  {h['title']}")
    print()

  keyword    documents: ['DOC-003', 'DOC-001', 'DOC-004']
  embeddings documents: ['DOC-003', 'DOC-007', 'DOC-005']
  same set? False

  probe: 'how long does delivery to the nordics take'

  keyword-tfidf
      DOC-003  score=0.7201  Delivery timescales by region
      DOC-001  score=0.2167  Returns and refunds policy
      DOC-005  score=0.1080  Billing, invoices and payment terms

  databricks-embeddings
      DOC-003  score=0.8263  Delivery timescales by region
      DOC-005  score=0.5775  Billing, invoices and payment terms
      DOC-007  score=0.5004  Bulk and fit-out orders



### A paraphrase that has no words in common

In [0]:
probe2 = "I changed my mind, can I send it back?"
print(f"  probe: {probe2!r}")
print("  (the documents never use the phrase 'send it back')\n")
for r in (kw, emb):
    print(f"  {r.name}")
    for h in r.search(probe2, k=3, audience="customer"):
        print(f"      {h['doc_id']}  score={h['score']:.4f}  {h['title']}")
    print()

  probe: 'I changed my mind, can I send it back?'
  (the documents never use the phrase 'send it back')

  keyword-tfidf
      DOC-005  score=0.1147  Billing, invoices and payment terms
      DOC-001  score=0.0000  Returns and refunds policy
      DOC-003  score=0.0000  Delivery timescales by region

  databricks-embeddings
      DOC-001  score=0.4954  Returns and refunds policy
      DOC-007  score=0.4477  Bulk and fit-out orders
      DOC-005  score=0.3369  Billing, invoices and payment terms



## 7. The uncomfortable finding

Two things are true at once:

1. **The swap was free.** `core.py` did not change. The Protocol held.
2. **You cannot tell which retriever is better by reading the answers.**

The second is the problem. A good model papers over mediocre retrieval — it takes
three documents, finds the relevant sentence in one of them, and writes a confident,
correct-sounding answer. The answer looks the same whether the other two documents
were useful or noise.

**So how would you ever know?** Not by reading outputs. You need a measurement.

| Question | Answered in |
|---|---|
| How do I *measure* retrieval quality? | [Session 6](#) — `retrieval_relevance` |
| What did the baseline actually score? | **0.20.** Four of five retrieved documents were irrelevant. |
| Did the correct answers hide it? | Yes. `correctness` was **1.00** at the same time. |

> **Carry this forward.** An agent can be right for the wrong reasons, at scale,
> and look perfect in a demo. The only defence is measuring the steps, not the
> output.

### Try this before you move on

1. Write a third retriever that returns the **first three documents, always**,
   ignoring the query. Swap it in. Read the answer. Is it obviously wrong?
2. Set `k=1` on each retriever and re-run cell 4. Which one degrades first?
3. Re-read cell 6's scores. The keyword retriever gives a **higher** score to a
   document it ranks third than the embedding retriever gives to its first.
   **Scores from different retrievers are not comparable** — a fact that breaks
   naive relevance thresholds, as [Lab 6B](#) demonstrates.